# Agent Brainstorm

One issue is that agents are really good at data harmonizing, basically. But it's not reproducible, which matters for sensitive research data. So how can we use agents in a reproducible manner?

## Calling llama3.1:8b through Ollama

Before running these cells:
- Start Ollama and pull the model: `ollama pull llama3.1:8b` (about 5 GB).
- Use the `~/venvs/harmonize` interpreter as the kernel, so `hbench` and its dependencies import.

Every call goes through `hbench.llm.LLMClient` inside a run opened with `start_run`, as the repo rules require. The raw log goes to `~/hbench_raw/` (or `$HBENCH_RAW`), and the temperature is set explicitly.

`llama3.1:8b` is not in `env/engine-lock.json` yet, so the run header records engine drift and `hbench log validate` will flag the run. That is fine for brainstorming; agree with the team before adding the model to the lock.

In [1]:
import json
import sys
import urllib.request
from pathlib import Path

REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO))  # use this checkout's hbench

from hbench.design import make_condition
from hbench.llm import LLMClient
from hbench.runlog.session import raw_root, start_run

MODEL = "llama3.1:8b"
OLLAMA = "http://localhost:11434"

In [2]:
# Is Ollama running, and has the model been pulled?
with urllib.request.urlopen(f"{OLLAMA}/api/version", timeout=5) as r:
    print("ollama", json.load(r)["version"])
with urllib.request.urlopen(f"{OLLAMA}/api/tags", timeout=5) as r:
    local = [m["name"] for m in json.load(r)["models"]]
assert MODEL in local, f"{MODEL} not pulled; run: ollama pull {MODEL}  (have: {local})"

ollama 0.40.0


In [3]:
# Open a logged run. cold_start unloads the model first, so every run starts from the same engine state.
# The run holds a lease on Ollama until log.close() at the end of the notebook.
cond = make_condition(
    method={"name": "agent_brainstorm"},
    model={"name": MODEL},
    decoding={"temperature": 0, "seed": 1},
    engine={"transport": "ollama", "protocol": "cold_start"},
    data={"study": "clark"},
    replicate=1,
)
log = start_run(raw_root(), "agent_brainstorm", cond, task={"kind": "sm"}, models=[MODEL])
client = LLMClient(log, MODEL)  # takes temperature and seed from cond
print("run folder:", log.run_dir)

run folder: /home/jennifer/hbench_raw/agent_brainstorm-agent_brainstorm_llama3.1_8b_T0_s1_r1-clark-20261109T183754Z-8672


In [4]:
# A plain chat call
res = client.chat(
    [{"role": "user", "content": "In one sentence: what does the GDC field `primary_site` record?"}],
    caller="notebook:chat",
)
print(res.text)
print(f"{res.latency_s:.1f}s, prompt tokens: {res.prompt_tokens}")

The GDC field `primary_site` records the primary site where the sample was collected from.
26.7s, prompt tokens: 26


In [5]:
# A JSON answer, with the decision named so hbench records how close it was to flipping
res = client.chat(
    [{"role": "user", "content": (
        'Source column: "Tumor_Site". Candidate GDC columns: primary_site, site_of_resection_or_biopsy, '
        'tissue_or_organ_of_origin, tumor_depth_descriptor.\n'
        'Answer with JSON only: {"target": "<one candidate, or NONE>"}')}],
    caller="notebook:json",
    format="json",
    decision={"type": "json_field", "field": "target"},
)
print(res.text)
print(res.evidence())

{
  "target": "primary_site"
}
{'span_id': 'driver:4', 'prompt_sha256': '72d5965ea52c6771df70ce68051757da53d2a00b3e49159c2c74c1716990e5f3', 'margin': 0.04192918539047241, 'margin_value': 0.7533225119113922, 'margin_def': 'prefix/1', 'margin_pos': 0, 'runner_up': '{"', 'margin_found': True, 'margin_note': None, 'n_constrained': 0, 'p_path_default': {'/api/chat': {'p': 0.35460923841040154, 'exact': False, 'lower_bound': True}, '/v1/chat/completions': {'p': 0.2688646093321562, 'exact': False, 'lower_bound': True}}}


In [6]:
# A tool call: the model chooses a tool and its arguments instead of answering in text
tools = [{
    "type": "function",
    "function": {
        "name": "get_column_values",
        "description": "Return up to 5 sample values of a source column.",
        "parameters": {
            "type": "object",
            "properties": {"column": {"type": "string", "description": "Source column name"}},
            "required": ["column"],
        },
    },
}]
res = client.chat(
    [{"role": "user", "content": "Before matching the source column Tumor_Site to GDC, look at its values."}],
    caller="notebook:tools",
    tools=tools,
    decision={"type": "tool_call", "field": "arguments.column"},
)
print("tool calls:", res.tool_calls)
print("text:", res.text)

tool calls: [{'name': 'get_column_values', 'arguments': {'column': 'Tumor_Site'}}]
text: 


In [7]:
# Close the run when you are done: this releases the lease and writes the engine audit.
log.close()
print(f"check it with: ~/venvs/harmonize/bin/python -m hbench log validate {log.run_dir}")

check it with: ~/venvs/harmonize/bin/python -m hbench log validate /home/jennifer/hbench_raw/agent_brainstorm-agent_brainstorm_llama3.1_8b_T0_s1_r1-clark-20261109T183754Z-8672
